# Selenium Tutorial - Lecture Version

This tutorial is accompanying the lecture slides for "Getting data from the Web (Part 2).

**Author**: Eni Mustafaraj   
**Created on:** Sep 10, 2026

**Table of Contents**

1. [Using `requests` and `BeautifulSoup`](#sec1)
2. [Simple use of `seleniumbase`](#sec2)
3. [Clicking button with `seleniumbase`](#sec3)
4. [Scrolling with `seleniumbase`](#sec4)
5. [Your task: Simple parser](#sec5)
6. [Handling pagination](#sec6)
7. [Your task: Modify parser for mockup page](#sec7)

<h2 id="sec1">1. Using <em>requests</em> and <em>BeautifulSoup</em></h2>

We will first try to get the content of both our pages using the two libraries we already know.

In [2]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://cs.wellesley.edu/~cs315/scraping"

We'll write a function that retrieves the pages, given the URL.

In [3]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url)

    print(f"URL: {url}")
    print(f"Status Code: {response.status_code}")

    if response.status_code == 200:
        return response.text
    else:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None

Let's test that the function works as intended:

In [4]:
#test to see if working 
url = f"{BASE_URL}/page1.html"
page = fetch_page_content(url)
print(page)

URL: https://cs.wellesley.edu/~cs315/scraping/page1.html
Status Code: 200
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Data Page V1</title>
    <style>
        table, th, td { 
            border: 1px solid black; 
            border-collapse: collapse; 
            padding: 8px; }
        th { background-color: #f2f2f2; }
    </style>
</head>
<body>
    <h1>Student Grades V1</h1>
    <table id="grades-table">
        <thead>
            <tr>
                <th>Student ID</th>
                <th>Name</th>
                <th>Grade</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>101</td>
                <td>Alice Johnson</td>
                <td>A</td>
            </tr>
            <tr>
                <td>102</td>
                <td>Bob Smith</td>
                <td>B</td>
            </tr>
            <tr>
                <td>103</td>
                <td>Charlie Brown</td>
                <td>A-</td>
   

And now let's write a function that will extract the values of the table:

In [5]:
def parse_table(html_content):
    """Parses HTML content and returns table rows as a list of tuples."""
    if not html_content:
        return []

    soup = BeautifulSoup(html_content, "html.parser")
    rows = soup.find_all("tr")

    table_data = []
    for row in rows:
        cells = [cell.get_text(strip=True) for cell in row.find_all(["th", "td"])]
        if cells:
            table_data.append(tuple(cells))

    return table_data

Now let's test this function with the output of the `fetch_page_content`:

In [6]:
parse_table(page)

[('Student ID', 'Name', 'Grade'),
 ('101', 'Alice Johnson', 'A'),
 ('102', 'Bob Smith', 'B'),
 ('103', 'Charlie Brown', 'A-')]

Let's now try these two functions with the dynamic page:

In [7]:
url = f"{BASE_URL}/page2.html"
page = fetch_page_content(url)
print(page)
parse_table(page)

URL: https://cs.wellesley.edu/~cs315/scraping/page2.html
Status Code: 200
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Data Page V2</title>
    <style>
        table, th, td { 
            border: 1px solid black; 
            border-collapse: collapse;
            padding: 8px; }
        th { background-color: #e2e2e2; }
    </style>
</head>
<body>
    <h1>Student Grades V2</h1>
    
    <div id="table-container"></div>

    <script src="data.js"></script>
</body>
</html>


[]

**Conclusion:** For the 2nd page, where the table is injected with JS, our code didn't return any rows. In order to get the entire content of the page, we will use Selenium, which opens a broswer and creates the DOM. 

<h2 id="sec2">2. Simple use of <em>seleniumbase</em></h2>

Seleniumbase is a framework built on top of standard Selenium. It does everything that Selenium does but in a better way and with less overhead.

First, let's open the URL on the browser, load the page, then save the rendered content:

In [8]:
pip install seleniumbase

Note: you may need to restart the kernel to use updated packages.


In [9]:
from seleniumbase import Driver

url = f"{BASE_URL}/page2.html"

with Driver() as driver:
    driver.open(url)
    # this line is here to make sure the table is created before we save the page
    driver.wait_for_element("#grades-table") 
    driver.sleep(5) # sleep so that we can see the browser
    rendered_html = driver.get_page_source()

Now that we have the rendered HTML, we can use the function from above to parse the HTML.

In [10]:
data = parse_table(rendered_html)

print(f"Extracted {len(data)} rows:")
for row in data:
    print(row)


Extracted 4 rows:
('Student ID', 'Name', 'Grade')
('201', 'Diana Prince', 'A+')
('202', 'Evan Wright', 'B+')
('203', 'Fiona Gallagher', 'A')


In [11]:
print(rendered_html)

<html lang="en"><head>
    <meta charset="UTF-8">
    <title>Data Page V2</title>
    <style>
        table, th, td { 
            border: 1px solid black; 
            border-collapse: collapse;
            padding: 8px; }
        th { background-color: #e2e2e2; }
    </style>
</head>
<body>
    <h1>Student Grades V2</h1>
    
    <div id="table-container">
        <table id="grades-table">
            <thead>
                <tr>
                    <th>Student ID</th>
                    <th>Name</th>
                    <th>Grade</th>
                </tr>
            </thead>
            <tbody>
    
            <tr>
                <td>201</td>
                <td>Diana Prince</td>
                <td>A+</td>
            </tr>
        
            <tr>
                <td>202</td>
                <td>Evan Wright</td>
                <td>B+</td>
            </tr>
        
            <tr>
                <td>203</td>
                <td>Fiona Gallagher</td>
                <td>A</

As we can see, through this method we were able to extract the table content from the dynamic page as well.

<a id="sec3"></a>

## 3. Clicking buttons with `seleniumbase`

Our [new interactive page](https://cs.wellesley.edu/~cs315/scraping/page3.html) requires us to click a button a few times for all the data to appear. We'll write code to do that. But first, let's look at the HTML source code to find the button:

```
<body>
    <h1>Student Grades (Interactive)</h1>

    <!-- Button to dynamically add more rows -->
    <button id="add-btn">Add More Students</button>

    <table id="grades-table">
        <thead>
```

First we will see what happens if we click once:

In [12]:
newURL = "https://cs.wellesley.edu/~cs315/scraping/page3.html"

with Driver() as driver:
    driver.open(newURL)
    button_selector = "#add-btn" # the id attribute value for the button
    driver.sleep(2)
    driver.click(button_selector)
    driver.sleep(2)

Since we wouldn't know in advance how much data there is and how many times we need to click the button, we have to use the functions of the library to understand when the button is disabled.

In [15]:
with Driver() as driver:
    driver.open(newURL)
    button_selector = "#add-btn" # the id attribute value for the button
    driver.sleep(1)
    while not driver.is_attribute_present(button_selector, "disabled"):
        driver.click(button_selector)
        driver.sleep(1)
        
    # this line needs to be withing the "with ... as ..." block
    rendered_html = driver.get_page_source()


Let's check that we got all the data by calling the scraping function:

In [16]:
parse_table(rendered_html)

[('Student ID', 'Name', 'Grade'),
 ('201', 'Diana Prince', 'A+'),
 ('202', 'Evan Wright', 'B+'),
 ('203', 'Fiona Gallagher', 'A'),
 ('204', 'George Clark', 'B'),
 ('205', 'Hannah Abbott', 'A-'),
 ('206', 'Ian Malcolm', 'A'),
 ('207', 'Julia Roberts', 'B+'),
 ('208', 'Kevin Bacon', 'A+'),
 ('209', 'Laura Croft', 'A-')]

<a id="sec4"></a>

## 4. Scrolling with `seleniumbase`

Very often, the server will only serve a portion of the page and then load more of the content based on user actions such as scrolling. In fact, most of the web today is based on the concept of "infinite scrolling", new content keeps showing up as you scroll down a page.

Seleniumbase makes it very easy to load the content that is updated dynamically while scrolling.

We will test this with our [infinte_dramas.html](https://cs.wellesley.edu/~cs315/scraping/infinite_dramas.html) page (although it's not infinite).

First we will simply get the page to see what's there:

In [17]:
from seleniumbase import Driver

url = "https://cs.wellesley.edu/~cs315/scraping/infinite_dramas.html"

with Driver() as driver:
    driver.open(url)
    driver.sleep(1)
    rendered_html = driver.get_page_source()

Let's write a few lines of BeautifulSoup to see how many dramas are there:

In [18]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(rendered_html, "html.parser")
cards = soup.find_all("div", class_="drama-card")
print(len(cards))

3


As expected, we got the 3 dramas that show in the page at the start. Now we need to **simulate** scrolling in order to ge the rest of the dramas show up.

In [19]:
from seleniumbase import Driver

url = "https://cs.wellesley.edu/~cs315/scraping/infinite_dramas.html"

with Driver() as driver:
    driver.open(url)
    driver.sleep(1)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    driver.sleep(1)
    rendered_html = driver.get_page_source()

Let's check again how many "drama cards" are received:

In [20]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(rendered_html, "html.parser")
cards = soup.find_all("div", class_="drama-card")
print(len(cards))

6


We know that there are more than 6 dramas, so we need to have a loop that keeps scrolling. That means that we need to use the scraping capabilities of selenium to check on the fly for the number of drama cards.

**Note:** This example is an excellent one for showing where the looping construct `while` instead of `for` is needed. We have to keep scrolling until we read the end and then break out of the loop.

In [21]:
from seleniumbase import Driver

url = "https://cs.wellesley.edu/~cs315/scraping/infinite_dramas.html"

last_count = 0

with Driver() as driver:
    driver.open(url)

    while True:
        # 1. Scroll to the bottom of the page using JavaScript
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        # 2. Wait for dynamic JavaScript content to render
        driver.sleep(1.0)
        
        # 3. Check the new count of target elements
        current_count = len(driver.find_elements(".drama-card"))
        print(f"Drama count: {current_count}")
        
        # 4. Exit the loop if no new items loaded after scrolling
        if current_count == last_count:
            print("Reached the bottom of the page.")
            break
            
        last_count = current_count

Drama count: 6
Drama count: 9
Drama count: 12
Drama count: 15
Drama count: 15
Reached the bottom of the page.


<a id="sec5"></a>

## 5. Your Task: Write a simple parser

Now that we were able to get the list of all dramas from our Top Dramas page, **write a function** similar to function `parse_table` above to exract the following fields from each drama card: 

```
{rank: "#6", 
title: "Alchemy of Souls", 
year: "2022", 
eps: "20 episodes", 
rating: "8.9", 
type: "Korean Drama", 
synopsis: "A powerful sorceress in a blind woman's body encounters a man from a prestigious family..." }
```
   


In [ ]:

def parse_drama(html_content, card_selector=".drama-card"):
    soup = BeautifulSoup(html_content, "html.parser")
    dramas = []

    for card in soup.select(card_selector):
        def grab(selector):
            el = card.select_one(selector)
            return el.get_text(strip=True) if el else None

        # meta format: "Korean Drama - 2022, 20 episodes"
        meta_text = grab(".meta") or ""
        dtype, year, eps = None, None, None
        if " - " in meta_text:
            dtype, rest = meta_text.split(" - ", 1)
            pieces = [p.strip() for p in rest.split(",")]
            year = pieces[0] if len(pieces) > 0 else None
            eps = pieces[1] if len(pieces) > 1 else None
            dtype = dtype.strip()

        # rating format: "★ 8.9"
        rating = grab(".rating")
        if rating:
            rating = rating.replace("★", "").strip()

        dramas.append({
            "rank": grab(".rank"),
            "title": grab(".title"),
            "year": year,
            "eps": eps,
            "rating": rating,
            "type": dtype,
            "synopsis": grab(".synopsis"),
        })

    return dramas

print (parse_drama(rendered_html))


[{'rank': '#1', 'title': 'When Life Gives You Tangerines', 'year': '2025', 'eps': '16 episodes', 'rating': '9.3', 'type': 'Korean Drama', 'synopsis': "A story that resembles a tribute to our parents' tender and still youthful seasons..."}, {'rank': '#2', 'title': 'Twinkling Watermelon', 'year': '2023', 'eps': '16 episodes', 'rating': '9.2', 'type': 'Korean Drama', 'synopsis': 'In 2023, high school student Eun Gyeol, a CODA with a passion for music, leads a double life...'}, {'rank': '#3', 'title': 'Move to Heaven', 'year': '2021', 'eps': '10 episodes', 'rating': '9.1', 'type': 'Korean Drama', 'synopsis': "Han Geu Ru is an autistic 20-year-old guy. He works for his father's business Move to Heaven..."}, {'rank': '#4', 'title': 'Hospital Playlist', 'year': '2020', 'eps': '12 episodes', 'rating': '9.1', 'type': 'Korean Drama', 'synopsis': 'Hospital Playlist tells the story of doctors and nurses working at Yulje Medical Center...'}, {'rank': '#5', 'title': 'Reply 1988', 'year': '2015', 'ep

In [ ]:
def parse_drama(html_content):
    """
    """

Once you have your function, call it on the html content of scrolled page and then save the list of dictionaries into a JSON file, `scraping_results.json`.

<a id="sec6"></a>

## 6. Handling Pagination

Our [mockup drama page](https://cs.wellesley.edu/~cs315/scraping/pagination-mockup.html) uses a pagination format, where the URL encodes the index of the page to visit, for example:

```
https://cs.wellesley.edu/~cs315/scraping/pagination-mockup.html?page=4
```

Below we show some sample code how to handle pagination.

In [32]:
import math
from seleniumbase import Driver

url = "https://cs.wellesley.edu/~cs315/scraping/pagination-mockup.html"

with Driver() as driver:
    driver.open(url)

    # 1. Extract total results count (e.g., "105 results" -> 105)
    total_text = driver.get_text("p.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 4. Loop through each page URL
    for page in range(1, total_pages + 1):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 5. Extract items on the current page
        cards = driver.find_elements(".box")
        print(f"Page {page}: Scraped {len(cards)} dramas")

Total results: 105 | Total pages: 6
Page 1: Scraped 20 dramas
Page 2: Scraped 20 dramas
Page 3: Scraped 20 dramas
Page 4: Scraped 20 dramas
Page 5: Scraped 20 dramas
Page 6: Scraped 5 dramas


<a id="sec7"></a>

## 7. Your Task: Modify Parser

You wrote a `parse_drama` function in Section 5. Update that function to extract the information from the `.box` elements in our paginated mockup drama page. Then calle that function within the loop from Section 6 to scrape the content of each drama and save everything into a JSON file.

In [ ]:
all_dramas = []

for page in range(1, NUM_PAGES + 1):      
    driver.get(f"{BASE_URL}?page={page}")
    driver.sleep(1)
    all_dramas.extend(parse_drama(driver.get_page_source(), card_selector=".box"))

print(f"Scraped {len(all_dramas)} dramas across {NUM_PAGES} pages")

with open("scraping_results.json", "w", encoding="utf-8") as f:
    json.dump(all_dramas, f, indent=2, ensure_ascii=False)